<div style="background: linear-gradient(120deg,#1e1b4b,#312e81); border-radius:16px; padding:36px 40px; color:#e2e8f0;">
<h1 style="margin:0; font-size:2.1em; color:#ffffff;">🌌 エンベディングとベクトルデータベース</h1>
<h3 style="margin:6px 0 0 0; font-weight:400; color:#93c5fd;">セッション 2 / 4 — 意味の幾何学</h3>
<p style="margin-top:18px; color:#cbd5e1;">NordWind Energy ワークショップシリーズ · Retrieval, Vectors & Graphs</p>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noctetemp/nordwind-workshop/blob/main/session2_vectors_ja.ipynb)

## 🗺️ 今日の道のり

前回、私たちのリトリーバーは**たった1回の行列積**で検索を行いました — クエリを*全*チャンクと比較する方法です。今日は、その行列積が本当は何を計算しているのかを突き止め、それが壁にぶつかる瞬間を目撃し、本物のインフラで置き換えます。

* **🧭 エンベディングとは実際何なのか** — 座標としての「意味」。ライブデモ: 言い換え文が無関係な文を約17倍のスコアで圧倒し……そして*正反対の指示*がそのどちらをも上回ります。類似度 ≠ 同意。
* **👁️ 意味の空間を見る** — 本日の「おおっ」ポイント: NordWind の全チャンクをインタラクティブな2次元マップに投影します。ポストモーテムはポストモーテムと、ランブックはランブックと勝手に集まります — *誰もモデルに教えていないのに*。さらにクエリをマップ上に落とし、その近傍が光る様子を見ます。
* **🧱 スケールの壁** — 総当たり検索をベンチマークし、500万チャンクに外挿し、そして **HNSW** に出会います: ベクトルDBが数百万のベクトルのうち数百件しか触らずに検索できる仕組み。
* **🗄️ 本物のベクトルデータベース** — Qdrant を*このノートブックの中で*動かします。コレクション、ペイロード、そして過小評価されがちな超能力: **メタデータフィルタリング**。
* **🔎 純粋なベクトル検索がつまずく場所** — `INC-2118` のような識別子。解決策: **ハイブリッド検索**(BM25 + ベクトルの融合)。
* **🎯 リランキング** — 2つ目の、より賢いモデルが上位20件を読み直して並べ替えます。安価に精度を買う方法。
* **🏗️ RAG v2** — すべてを配線して……最後にもう一度、*あの難問*を投げます。ネタバレ: 配管は良くなっても、壁は同じ場所にあります。😏

> すべてセッション1と同じ NordWind の世界で動きます — 同じ65ドキュメント、同じ20インシデント、同じアニメキャラのエンジニアたちです。

## 0 · セットアップ — 同じ世界、増える道具

In [ ]:
%pip -q install sentence-transformers qdrant-client umap-learn rank_bm25 plotly anthropic tiktoken
print("✅ ready")

In [ ]:
import json, urllib.request, pathlib, os
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/noctetemp/nordwind-workshop/main/dataset/documents.jsonl"
LOCAL = pathlib.Path("documents.jsonl")
if not LOCAL.exists():
    urllib.request.urlretrieve(DATA_URL, LOCAL)
docs = [json.loads(line) for line in LOCAL.read_text().splitlines()]

def chunk_text(text, size=800, overlap=150):
    chunks, i = [], 0
    while i < len(text):
        chunks.append(text[i:i+size]); i += size - overlap
    return chunks

chunks, meta = [], []
for d in docs:
    for j, ch in enumerate(chunk_text(d['text'])):
        chunks.append(ch)
        meta.append({"doc_id": d['doc_id'], "doc_type": d['doc_type'],
                     "title": d['title'], "date": d['date'], "chunk": j})
print(f"📚 {len(docs)} docs → {len(chunks)} chunks (same pipeline as Session 1)")

In [ ]:
# Anthropic client (same pattern as Session 1 — LLM cells degrade gracefully without a key)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass
import anthropic
MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic() if os.environ.get("ANTHROPIC_API_KEY") else None
def ask_claude(prompt, max_tokens=600):
    if client is None:
        return "[LLM skipped — no API key configured]"
    r = client.messages.create(model=MODEL, max_tokens=max_tokens,
                               messages=[{"role": "user", "content": prompt}])
    return r.content[0].text

---
## 1 · 🧭 エンベディングとは実際何なのか

エンベディングモデルは、拍子抜けするほどシンプルな目的で訓練されたニューラルネットワークです: **意味が近い文は近くに、そうでない文は遠くに配置せよ。** これを数十億のペアで訓練すると、ネットワークは*意味そのもの*の座標系を発明せざるを得なくなります。

任意のテキストに対する出力はベクトル — 私たちのモデルでは **384個の数値**です。個々の数値に意味はありません(「217次元目 = 怒り度」のようにはなっていません)。意味は空間の*方向*に宿り、全座標に分散しています。

近さは**コサイン類似度** — 2つのベクトルのなす角 — で測ります: `1.0` = 同じ方向(同じ意味)、`0` = 無関係、負 = 対立。全ベクトルを長さ1に正規化しているので、コサイン類似度は単なる内積です。空間を体感してみましょう:

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")   # 384 dims

pairs = [
    ("The payment gateway timed out during peak hours.",
     "Transactions were failing in the evening because the PSP was slow."),   # paraphrase, ~zero shared keywords
    ("The payment gateway timed out during peak hours.",
     "The billing engine computes invoices from meter readings."),            # same domain, different topic
    ("The payment gateway timed out during peak hours.",
     "Frieren finally learned to appreciate her time with Himmel."),          # unrelated
    ("Restart the service to clear the cache.",
     "Do NOT restart the service, the cache must be preserved."),             # ⚠️ opposite instructions!
]
for a, b in pairs:
    va, vb = embedder.encode([a, b], normalize_embeddings=True)
    print(f"{float(va @ vb):5.2f}  |  {a[:44]:44s} ↔  {b[:48]}")

4つの数値に3つの教訓が隠れています:

1. **言い換え**ペア(約0.5)は、無関係な文(約0.03)をキーワードほぼゼロ共有のまま圧倒します — セッション1の検索が機能した理由がこれです。(0.9ではなく0.5です — 現実の言い換えスコアは想像より低く出ます。重要なのは*差*であり、だからこそ閾値ではなくランキングを使うのです。)
2. 同ドメイン・別トピックは中間に落ちます — エンベディングは*何についての話か(aboutness)*を段階的に捉えます。
3. ⚠️ そして衝撃の事実: **「再起動せよ」と「再起動するな」が約0.8 — 表の中で最高スコア、真の言い換えペアより上です。** 両者は強烈に「同じことについて」話しており、類似度が測るのは *aboutness* — 同意でも、真実でも、論理でもありません。コンプライアンス規則や安全手順に純粋ベクトル検索を提案する人が現れた日には、この一枚を思い出してください。

さて — 384次元は可視化不能です。本当に?

---
## 2 · 👁️ 意味の空間を見る

**NordWind の全チャンク**をエンベディングし、UMAP で384次元を2次元に投影します。近傍関係をできる限り保つ手法です。1つ注意: 距離は歪みます(384次元を2次元に潰す以上、避けられません)— 信じるべきは*クラスタ*であって、正確な距離ではありません。

プロットはインタラクティブです: **ホバー**でチャンクを読み、クラスタに**ズーム**し、凡例クリックでドキュメント種別の表示を切り替えられます。

In [ ]:
E = embedder.encode(chunks, normalize_embeddings=True, show_progress_bar=True)
print("Embedding matrix:", E.shape)

In [ ]:
import umap
import pandas as pd
import plotly.express as px

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42).fit(E)
xy = reducer.embedding_          # keep the fitted reducer — we'll reuse it for the query

df = pd.DataFrame({
    "x": xy[:, 0], "y": xy[:, 1],
    "type": [m["doc_type"] for m in meta],
    "doc": [m["doc_id"] for m in meta],
    "text": [c[:150].replace("\n", " ") for c in chunks]})

fig = px.scatter(df, x="x", y="y", color="type", hover_name="doc", hover_data={"text": True, "x": False, "y": False},
    title="Every chunk of NordWind's knowledge, as a map of meaning",
    color_discrete_map={"postmortem": "#f87171", "runbook": "#34d399",
                        "adr": "#60a5fa", "slack_thread": "#f59e0b", "overview": "#a78bfa"},
    width=950, height=620)
fig.update_traces(marker=dict(size=7, opacity=0.85))
fig.show()

🎤 **スクロールする前に、立ち止まって見てください。** モデルはポストモーテムが何か、ランブックが何かを一度も教わっていません。生のテキストを読んだだけです — それでもドキュメント種別が目に見える領域を形成します。ポストモーテムは*ポストモーテムのような話し方*をするからです。混ざっている領域にズームしてみてください: ポストモーテム領域に座っているランブックが見つかるはずです。**同じインシデント**を扱っているからです — フォーマットよりトピックが勝つ。それが意味空間です。

そして本当の魔法: このマップ上では、検索とはただの**近接**です。クエリを落として、セッション1のリトリーバーが実際にやっていたことを光らせてみましょう:

In [ ]:
query = "Why did payments time out in the evening?"
qv = embedder.encode([query], normalize_embeddings=True)
scores = (E @ qv.T).ravel()
top10 = set(np.argsort(-scores)[:10])

# project the query into the SAME 2D map, reusing the fitted reducer (approximate by nature)
qxy = reducer.transform(qv)

import plotly.graph_objects as go
fig = go.Figure()
fig.add_scatter(x=xy[:,0], y=xy[:,1], mode="markers", name="all chunks",
                marker=dict(size=6, color="#cbd5e1"),
                text=[f"{m['doc_id']}: {c[:110]}" for m, c in zip(meta, chunks)], hoverinfo="text")
sel = sorted(top10)
fig.add_scatter(x=xy[sel,0], y=xy[sel,1], mode="markers", name="top-10 retrieved",
                marker=dict(size=11, color="#f87171"),
                text=[f"{meta[i]['doc_id']}: {chunks[i][:110]}" for i in sel], hoverinfo="text")
fig.add_scatter(x=qxy[:,0], y=qxy[:,1], mode="markers+text", name="THE QUERY",
                marker=dict(size=18, color="#111827", symbol="star"),
                text=["  ⭐ query"], textposition="middle right")
fig.update_layout(title="Retrieval is proximity: the query lands in payment territory",
                  width=950, height=620)
fig.show()

あの ⭐ があなたの質問です。知識と同じ空間に住んでいます。**「検索」は文字列マッチングであることをやめ、幾何学になりました。** 今日の残りすべて — インデックス、ハイブリッド、リランキング — はこの一枚の絵をめぐるエンジニアリングです。

---
## 3 · 🧱 スケールの壁 — そして HNSW の登り方

セッション1の検索は `E @ q` でした — クエリのたびに**全**ベクトルを触ります。正直に問いましょう: それはどれくらいまずいのか? 測って、外挿します:

In [ ]:
import time

def bench(n, dims=384, repeats=5):
    M = np.random.rand(n, dims).astype(np.float32)
    q = np.random.rand(dims).astype(np.float32)
    t0 = time.perf_counter()
    for _ in range(repeats):
        (M @ q).argmax()
    return (time.perf_counter() - t0) / repeats * 1000  # ms

for n in [len(chunks), 10_000, 100_000, 500_000]:
    print(f"{n:>9,} vectors → {bench(n):7.1f} ms per query (brute force)")

ms_500k = bench(500_000)
print(f"\nextrapolated:  5,000,000 vectors → ~{ms_500k*10:,.0f} ms per query")
print("...per query. Now multiply by queries-per-second, and add that the matrix no longer fits in RAM.")

総当たりは*線形*です: データ10倍、レイテンシ10倍、それが全クエリで発生します。実システムは **ANN — 近似最近傍(Approximate Nearest Neighbor)** インデックスを使い、その代表格が **HNSW**(*Hierarchical Navigable Small World*)です。

アイデアは美しく古典的: **グラフでできたスキップリスト**です。

- 各ベクトルが最近傍たちとリンクするグラフを作る(最下層 — 詳細で完全)
- その上に、どんどん疎になる層を積む — 各層は下の層の粗い「高速道路網」
- 検索時: 最上層から開始し、クエリへ向けて貪欲にホップし、1層降りて、繰り返す。国 → 都市 → 街路 → 番地、と絞り込む道案内と同じです。

ホップのたびに距離はおおむね半減し、数百万のうち**数百件**を触るだけで答えの近傍に到達します — O(N) ではなく **O(log N)**。代償は ANN の「A」: *近似*であること。まれに真の最近傍を取り逃します。あらゆるベクトルDBは再現率と速度を交換するノブ(`ef`、`M`)を公開しています — そのトレードオフこそが製品なのです。

In [ ]:
# A cartoon of HNSW's descent — illustrative, not the real algorithm
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
layers = [12, 40, 160]
target = np.array([0.82, 0.3])
entry  = np.array([0.08, 0.85])
for ax, n, name in zip(axes, layers, ["Layer 2 (sparse highways)", "Layer 1", "Layer 0 (all vectors)"]):
    pts = rng.random((n, 2))
    ax.scatter(pts[:,0], pts[:,1], s=14, color="#94a3b8")
    hops = np.linspace(0, 1, 4)[:, None]
    path = entry*(1-hops) + target*hops + rng.normal(0, 0.02, (4,2))
    ax.plot(path[:,0], path[:,1], "o-", color="#ef4444", lw=2, ms=6)
    ax.scatter(*target, s=140, color="#16a34a", marker="*", zorder=5)
    ax.set_title(name, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("HNSW: greedy hops on coarse layers, refine on dense ones — touch hundreds, not millions")
plt.tight_layout(); plt.show()

---
## 4 · 🗄️ 本物のベクトルデータベース — このノートブックの中に

**Qdrant** は Python 内で完全インメモリ動作します — セットアップゼロ、しかし正真正銘の本物です: 内部は HNSW で、本番クラスタに対して使うのと同じ API。ここでやることはすべて Milvus、Weaviate、pgvector、Pinecone に1対1で対応します — 概念は業界共通、構文がベンダー依存なだけです。

概念は2つ:
- **コレクション** = 次元数と距離指標が固定されたベクトルのテーブル
- 各**ポイント** = id + ベクトル + **ペイロード**(任意の JSON メタデータ)。エンジニアリング的な価値はペイロードに隠れています。

In [ ]:
from qdrant_client import QdrantClient, models

vdb = QdrantClient(":memory:")
vdb.create_collection("nordwind",
    vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE))

vdb.upsert("nordwind", points=[
    models.PointStruct(id=i, vector=E[i].tolist(),
                       payload={**meta[i], "text": chunks[i]})
    for i in range(len(chunks))])

print(vdb.get_collection("nordwind").points_count, "points indexed")

In [ ]:
def vsearch(query, k=5, flt=None):
    qv = embedder.encode([query], normalize_embeddings=True)[0].tolist()
    res = vdb.query_points("nordwind", query=qv, limit=k, query_filter=flt)
    return res.points

for p in vsearch("Why did payments time out in the evening?"):
    print(f"{p.score:.3f}  {p.payload['doc_id']:12s} {p.payload['title'][:58]}")

numpy 版と同じ結果です — そうでなくては困ります。ではペイロードの出番です。こう聞いてみましょう: *「支払いタイムアウトに**今すぐ対応する**には?」* 意味的にはポストモーテムもランブックも一致します。しかしユーザーが必要なのは**ランブック** — これは意味の区別ではなく、*メタデータ*の区別です:

In [ ]:
flt = models.Filter(must=[
    models.FieldCondition(key="doc_type", match=models.MatchValue(value="runbook"))])

print("── unfiltered (mixed intent) ──")
for p in vsearch("how to respond to payment timeouts right now", 3):
    print(f"{p.score:.3f}  [{p.payload['doc_type']:12s}] {p.payload['title'][:55]}")
print("\n── doc_type = runbook (operational intent) ──")
for p in vsearch("how to respond to payment timeouts right now", 3, flt):
    print(f"{p.score:.3f}  [{p.payload['doc_type']:12s}] {p.payload['title'][:55]}")

**フィルタリングは RAG で最も過小評価されている機能です。** ユーザーごとのアクセス制御、日付範囲、プロダクト領域、言語 — 本番では、ほぼすべての検索呼び出しがフィルタを伴います。しかも重要なことに、Qdrant は HNSW の走査*中*にフィルタします。top-k を取った*後*にフィルタする方式では、結果が0件になることがあるのです。

---
## 5 · 🔎 純粋なベクトル検索がつまずく場所 — そしてハイブリッド検索

エンベディングは*言語*で訓練されています。識別子 — `INC-2118`、`ISO 20022`、エラーコード、チケット番号 — は言語ではなく、モデルにとってほぼ無意味な記号です。特定のインシデントを ID で尋ねると、ベクトル検索は*似た響きの別インシデント*を自信満々に返します。一方、1970年代の技術(**BM25**、古典的検索エンジンの心臓部であるキーワードスコアリング)は一発で当てます — ただし、正しくトークナイズすれば:

In [ ]:
import re
from rank_bm25 import BM25Okapi

# tokenization matters! naive .split() leaves punctuation glued to words,
# so "INC-2118," never matches "INC-2118". \w+ extraction fixes it.
tok = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tok(c) for c in chunks])

def bm25_top(query, k=5):
    s = bm25.get_scores(tok(query))
    return [(s[i], i) for i in np.argsort(-s)[:k]]

q = "INC-2118 root cause"
print("── vector search ──")
for p in vsearch(q, 4):
    print(f"{p.score:.3f}  {p.payload['doc_id']:12s} {p.payload['title'][:55]}")
print("\n── BM25 keyword search ──")
for s, i in bm25_top(q, 4):
    print(f"{s:6.2f}  {meta[i]['doc_id']:12s} {meta[i]['title'][:55]}")

どちらかが常に優れているわけではありません — 言い換えにはベクトルが、正確な記号にはキーワードが勝ちます。本番システムは**両方**を走らせ、ランキングを融合します。標準の融合方式は **RRF**(*Reciprocal Rank Fusion*): 各リストでの順位から `1/(60+rank)` を計算して合算するだけ。チューニング不要、スコア正規化の悩みも無縁 — 拍子抜けするほど単純で、なかなか負けません:

In [ ]:
def hybrid(query, k=5, pool=20):
    vec_ids = [p.id for p in vsearch(query, pool)]
    kw_ids  = [i for _, i in bm25_top(query, pool)]
    rrf = {}
    for lst in (vec_ids, kw_ids):
        for rank, i in enumerate(lst):
            rrf[i] = rrf.get(i, 0) + 1.0 / (60 + rank)
    return sorted(rrf, key=rrf.get, reverse=True)[:k]

print(f"── hybrid (RRF) for: {q!r} — best of both worlds ──")
for i in hybrid(q, 5):
    print(f"  {meta[i]['doc_id']:12s} {meta[i]['title'][:58]}")

---
## 6 · 🎯 リランキング — 上位20件への「セカンドオピニオン」

私たちのエンベディングモデル(**バイエンコーダ**)は、クエリとドキュメントを*別々に*エンベディングします — だからこそ数百万チャンクのインデックスが可能なのですが、クエリとチャンクは内積の瞬間まで実際には*出会いません*。**クロスエンコーダ**はクエリとチャンクを**一緒に**読み、両者をまたいで Attention をかけます — はるかに正確ですが、数百万件に対して走らせるにはあまりに遅い。

そこで本番のパターン: **安いモデルが約20件の候補を取得 → 高いモデルが読み直して並べ替え → 上位5件を採用。** 精度は効く場所にだけ、コストは一握りの件数にだけ:

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # ~80MB

def rerank(query, candidate_ids, k=5):
    scores = reranker.predict([(query, chunks[i]) for i in candidate_ids])
    order = np.argsort(-scores)
    return [(scores[j], candidate_ids[j]) for j in order[:k]]

q2 = "what should we monitor to catch certificate problems before they hurt us?"
cands = hybrid(q2, k=20, pool=30)
print("── after rerank ──")
for s, i in rerank(q2, cands):
    print(f"{s:6.2f}  {meta[i]['doc_id']:12s} {meta[i]['title'][:58]}")

---
## 7 · 🏗️ RAG v2 — 本番形状のパイプライン

組み上げます: **ハイブリッド検索 → リランク → 生成。** これは本格的な RAG 製品が実際に動かしている構成にかなり近いものです(キャッシュ、評価、アクセス制御フィルタを足せば到達です):

In [ ]:
def rag_v2(question, k=5):
    cands = hybrid(question, k=20, pool=30)
    top = rerank(question, cands, k)
    context = "\n\n---\n\n".join(
        f"[Source: {meta[i]['doc_id']} — {meta[i]['title']}]\n{chunks[i]}" for _, i in top)
    print("Retrieved:", ", ".join(meta[i]['doc_id'] for _, i in top), "\n")
    return ask_claude(f"Answer using ONLY the sources below. Cite source ids. "
                      f"If insufficient, say so.\n\n{context}\n\nQuestion: {question}")

print(rag_v2("We keep having certificate-related outages. What happened before and what should we monitor?"))

検索は改善し、引用は明確になり、識別子にも頑健で、意図のフィルタリングもできる。RAG v2 は文句なしに強力なシステムです。では……最後に1つだけ。どの質問が来るか、もうお分かりですね。😏

In [ ]:
HARD_QUESTION = ("Which engineers have responded to incidents affecting services "
                 "that depend on payment-gateway? List all of them.")
print(rag_v2(HARD_QUESTION, k=6))

名前を数えてください。セッション1の正解(エンジニア10名)と比べてください。配管が良くなったぶん、拾える*断片*は増えました — それでも「payment-gateway に依存する」が billing-engine を*意味する*ことは知りようがなく、該当インシデントを**すべて**見つけたかどうかも知りようがなく、**JOIN** はできないままです。全コンポーネントをアップグレードしても壁は1ミリも動きませんでした。壁はコンポーネントの中にはないからです。**壁は幾何学の中にあります: 空間の点と点の間に、エッジは存在しない。**

## 🏁 今日学んだこと

- エンベディングは *aboutness* を座標として符号化する。コサイン類似度がその物差し — ただし測るのは aboutness であって真実ではない
- 意味空間をこの目で**見た**。検索は近接になった
- 総当たりは O(N)。**HNSW** は再現率をわずかに差し出して O(log N) を手に入れる
- ペイロード**フィルタリング**、BM25+ベクトルの**ハイブリッド**(RRF)、クロスエンコーダによる**リランキング** — デモと製品を分かつ3つのアップグレード
- そして: どれほどベクトルを磨いても *JOIN* はできない

---

<div style="background: linear-gradient(120deg,#052e16,#14532d); border-radius:16px; padding:30px 36px; color:#e2e8f0;">
<h2 style="margin:0; color:#ffffff;">⏭️ 次回予告…</h2>
<p style="font-size:1.05em; margin-top:14px;">空間の点にエッジはありません。しかし NordWind の知識には<i>エッジがある</i> — <b>owns</b>、<b>depends-on</b>、<b>responded-to</b> — セッション1で、私たちは気づかぬままそれを使って計算していました。</p>
<p style="font-size:1.05em;">次回: 関係(リレーションシップ)が第一級市民であるデータベースです。クエリを ASCII アートで描き — <code>(:Service)-[:DEPENDS_ON]->(:Service)</code> — そして会社全体が生きて動く、ドラッグ可能なネットワークとして目の前に現れます。あの難問は<b>1行</b>で答えが出ます。</p>
<h3 style="color:#86efac; margin-bottom:0;">セッション3: グラフデータベース — <i>あなたの知識には形がある</i> 🕸️</h3>
</div>